In [1]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


## Train

### Load

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import joblib
import json

RANDOM_STATE = 42
N_FOLDS = 5

df = pd.read_csv("train-test.csv", parse_dates=["date"])
print(f"Loaded {len(df)} rows, {df.shape[1]} cols")

Loaded 48000 rows, 14 cols


### EDA

In [3]:
df.head(4)

,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,0.95684,2.39595,645.41
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,2025-01-01,0.97623,2.43355,679.97
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,2025-01-01,1.00971,1.84491,1802.54
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,2025-01-01,0.94518,1.87712,1827.28


We can drop 'pickup' and 'delivery' column since they represent the data same as 'pickup_lat','pickup_lot' and 'delivery_lat','delivery_lon'

In [4]:
# Check for missing values
df.isna().sum()

load_id           0
pickup            0
delivery          0
pickup_lat        0
pickup_lon        0
delivery_lat      0
delivery_lon      0
distance          0
equipment         0
weight          300
date              0
market_index    374
quote_signal      0
posted_rate       0
dtype: int64

We will impute weight and market_index in feature engineering

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48000 entries, 0 to 47999
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   load_id       48000 non-null  object        
 1   pickup        48000 non-null  object        
 2   delivery      48000 non-null  object        
 3   pickup_lat    48000 non-null  float64       
 4   pickup_lon    48000 non-null  float64       
 5   delivery_lat  48000 non-null  float64       
 6   delivery_lon  48000 non-null  float64       
 7   distance      48000 non-null  float64       
 8   equipment     48000 non-null  object        
 9   weight        47700 non-null  float64       
 10  date          48000 non-null  datetime64[ns]
 11  market_index  47626 non-null  float64       
 12  quote_signal  48000 non-null  float64       
 13  posted_rate   48000 non-null  float64       
dtypes: datetime64[ns](1), float64(9), object(4)
memory usage: 5.1+ MB


In [6]:
df.describe(include='all')

,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
count,48000,48000,48000,48000.000000,48000.000000,48000.000000,48000.000000,48000.000000,48000,47700.000000,48000,47626.000000,48000.000000,48000.000000
unique,48000,64,64,NaN,NaN,NaN,NaN,NaN,3,NaN,NaN,NaN,NaN,NaN
top,TR-000001,Oklahoma City,Lexington,NaN,NaN,NaN,NaN,NaN,Dry Van,NaN,NaN,NaN,NaN,NaN
freq,1,1242,1197,NaN,NaN,NaN,NaN,NaN,27202,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,35.647545,-90.928964,35.641175,-90.857310,1135.856654,NaN,31028.844004,2025-05-31 19:53:29.400000,1.083387,2.062468,2373.980682
min,NaN,NaN,NaN,28.357650,-121.698490,28.357650,-121.698490,70.000000,NaN,-47500.000000,2025-01-01 00:00:00,0.676390,0.692280,57.220000
25%,NaN,NaN,NaN,31.986910,-98.400590,31.986910,-98.400590,550.400000,NaN,25800.000000,2025-03-18 00:00:00,0.949670,1.891030,1251.555000
50%,NaN,NaN,NaN,35.294790,-88.089150,35.294790,-87.528710,953.300000,NaN,31436.500000,2025-05-31 00:00:00,1.055800,2.055750,2030.760000
75%,NaN,NaN,NaN,39.411040,-83.285060,39.411040,-83.285060,1645.525000,NaN,37018.000000,2025-08-15 00:00:00,1.219590,2.221685,3330.750000
max,NaN,NaN,NaN,44.302960,-69.500000,44.302960,-69.500000,3439.800000,NaN,47500.000000,2025-10-31 00:00:00,1.467780,3.610350,25533.000000


- Equipment is object, we will one-hot encode it. 
- Also we will convert date to day_of_week, day_of_month, month
- Since year is always 2025 we don't need it
- distance is skewed, we will log transform it
- posted_rate is also skewed, we will log transform it also

In [7]:
df.equipment.value_counts()

equipment
Dry Van    27202
Reefer     12045
Flatbed     8753
Name: count, dtype: int64

### Feature Engineering

In [8]:

def engineer_features(df):
    df = df.copy()
    df["day_of_week"] = df["date"].dt.dayofweek
    df["day_of_month"] = df["date"].dt.day
    df["month"] = df["date"].dt.month
    df["log_distance"] = np.log1p(df["distance"])
    df = pd.get_dummies(df, columns=["equipment"], prefix="equip")
    return df

df_feat = engineer_features(df)


In [9]:
df_feat.head(4)

,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,weight,date,market_index,quote_signal,posted_rate,day_of_week,day_of_month,month,log_distance,equip_Dry Van,equip_Flatbed,equip_Reefer
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,30658.0,2025-01-01,0.95684,2.39595,645.41,2,1,1,5.617861,True,False,False
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,17555.0,2025-01-01,0.97623,2.43355,679.97,2,1,1,5.640132,False,False,True
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,31721.0,2025-01-01,1.00971,1.84491,1802.54,2,1,1,6.876058,True,False,False
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,32333.0,2025-01-01,0.94518,1.87712,1827.28,2,1,1,6.873578,True,False,False


IMPUTATION 

In [10]:

# (median chosen over mean since posted_rate/weight/market_index are skewed)
weight_median = df_feat["weight"].median()
market_index_median = df_feat["market_index"].median()
df_feat["weight"] = df_feat["weight"].fillna(weight_median)
df_feat["market_index"] = df_feat["market_index"].fillna(market_index_median)

# flag rows that were imputed - lets the model learn if missingness itself matters
df_feat["weight_was_missing"] = df["weight"].isna().astype(int)
df_feat["market_index_was_missing"] = df["market_index"].isna().astype(int)

In [11]:
df_feat.head(4)

,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,weight,date,...,posted_rate,day_of_week,day_of_month,month,log_distance,equip_Dry Van,equip_Flatbed,equip_Reefer,weight_was_missing,market_index_was_missing
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,30658.0,2025-01-01,...,645.41,2,1,1,5.617861,True,False,False,0,0
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,17555.0,2025-01-01,...,679.97,2,1,1,5.640132,False,False,True,0,0
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,31721.0,2025-01-01,...,1802.54,2,1,1,6.876058,True,False,False,0,0
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,32333.0,2025-01-01,...,1827.28,2,1,1,6.873578,True,False,False,0,0


In [12]:
df_feat.isnull().sum()

load_id                     0
pickup                      0
delivery                    0
pickup_lat                  0
pickup_lon                  0
delivery_lat                0
delivery_lon                0
distance                    0
weight                      0
date                        0
market_index                0
quote_signal                0
posted_rate                 0
day_of_week                 0
day_of_month                0
month                       0
log_distance                0
equip_Dry Van               0
equip_Flatbed               0
equip_Reefer                0
weight_was_missing          0
market_index_was_missing    0
dtype: int64

### Training

Assemble feature matrix

In [13]:
drop_cols = ["load_id", "pickup", "delivery", "date", "posted_rate", "distance"]
feature_cols = [c for c in df_feat.columns if c not in drop_cols]

X = df_feat[feature_cols]
y_raw = df_feat["posted_rate"].values
y = np.log1p(y_raw)  # log-transform target as decided

print(f"\nUsing {len(feature_cols)} features:")
print(feature_cols)
df_feat[feature_cols].head(4)


Using 16 features:
['pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'weight', 'market_index', 'quote_signal', 'day_of_week', 'day_of_month', 'month', 'log_distance', 'equip_Dry Van', 'equip_Flatbed', 'equip_Reefer', 'weight_was_missing', 'market_index_was_missing']


,pickup_lat,pickup_lon,delivery_lat,delivery_lon,weight,market_index,quote_signal,day_of_week,day_of_month,month,log_distance,equip_Dry Van,equip_Flatbed,equip_Reefer,weight_was_missing,market_index_was_missing
0,38.09122,-76.78906,38.16908,-72.74564,30658.0,0.95684,2.39595,2,1,1,5.617861,True,False,False,0,0
1,38.09122,-76.78906,39.22317,-72.96710,17555.0,0.97623,2.43355,2,1,1,5.640132,False,False,True,0,0
2,39.22317,-72.96710,44.30296,-87.52871,31721.0,1.00971,1.84491,2,1,1,6.876058,True,False,False,0,0
3,39.55328,-72.18051,34.84933,-86.28940,32333.0,0.94518,1.87712,2,1,1,6.873578,True,False,False,0,0


In [14]:
## Lets use KFold cross-validation to evaluate the model performance and avoid overfitting. We will use 5 folds and shuffle the data before splitting.
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

oof_pred_log = np.zeros(len(X))
fold_metrics = []
models = []

xgb_params = dict(
    n_estimators=1000,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    early_stopping_rounds=100,
    eval_metric="mae",
)

for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    model = xgb.XGBRegressor(**xgb_params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    val_pred_log = model.predict(X_val)
    oof_pred_log[val_idx] = val_pred_log

    # convert back to dollar scale for metrics
    val_pred_dollars = np.expm1(val_pred_log)
    val_true_dollars = np.expm1(y_val)

    mae = mean_absolute_error(val_true_dollars, val_pred_dollars)
    rmse = np.sqrt(mean_squared_error(val_true_dollars, val_pred_dollars))
    mape = np.mean(np.abs((val_true_dollars - val_pred_dollars) / val_true_dollars)) * 100
    r2 = r2_score(val_true_dollars, val_pred_dollars)

    fold_metrics.append(dict(fold=fold, mae=mae, rmse=rmse, mape=mape, r2=r2,
                              best_iteration=model.best_iteration))
    models.append(model)
    print(f"Fold {fold}: MAE=${mae:,.2f}  RMSE=${rmse:,.2f}  MAPE={mape:.2f}%  R2={r2:.4f}  "
          f"(best_iter={model.best_iteration})")

## Overall metrics on the out-of-fold predictions
oof_pred_dollars = np.expm1(oof_pred_log)
oof_true_dollars = y_raw

overall_mae = mean_absolute_error(oof_true_dollars, oof_pred_dollars)
overall_rmse = np.sqrt(mean_squared_error(oof_true_dollars, oof_pred_dollars))
overall_mape = np.mean(np.abs((oof_true_dollars - oof_pred_dollars) / oof_true_dollars)) * 100
overall_r2 = r2_score(oof_true_dollars, oof_pred_dollars)

print(f"Overall MAE:  ${overall_mae:,.2f}")
print(f"Overall RMSE: ${overall_rmse:,.2f}")
print(f"Overall MAPE: {overall_mape:.2f}%")
print(f"Overall R2:   {overall_r2:.4f}")


Fold 1: MAE=$99.93  RMSE=$528.48  MAPE=4.67%  R2=0.8694  (best_iter=278)
Fold 2: MAE=$104.44  RMSE=$610.09  MAPE=4.71%  R2=0.8305  (best_iter=300)
Fold 3: MAE=$107.86  RMSE=$615.45  MAPE=5.14%  R2=0.8304  (best_iter=300)
Fold 4: MAE=$105.97  RMSE=$615.86  MAPE=4.80%  R2=0.8343  (best_iter=295)
Fold 5: MAE=$102.77  RMSE=$590.87  MAPE=4.75%  R2=0.8406  (best_iter=367)
Overall MAE:  $104.19
Overall RMSE: $593.08
Overall MAPE: 4.82%
Overall R2:   0.8408


In [15]:

importances = np.mean([m.feature_importances_ for m in models], axis=0)
imp_df = pd.DataFrame({"feature": feature_cols, "importance": importances})
imp_df = imp_df.sort_values("importance", ascending=False)
print("\nfeatures importance:")
print(imp_df.to_string(index=False))



features importance:
                 feature  importance
            log_distance    0.681207
            delivery_lon    0.100149
              pickup_lon    0.071830
            quote_signal    0.047844
           equip_Dry Van    0.026181
            equip_Reefer    0.015073
              pickup_lat    0.014980
           equip_Flatbed    0.011152
                   month    0.008914
            delivery_lat    0.006908
                  weight    0.003986
            market_index    0.003337
market_index_was_missing    0.002408
            day_of_month    0.002251
             day_of_week    0.001969
      weight_was_missing    0.001809


In [16]:

# Taking Best n_estimators from the median of best_iteration across folds to train the final model on all data. This is a common practice to avoid overfitting and ensure that the final model is trained with a reasonable number of trees.
best_n_estimators = int(np.median([m.best_iteration for m in models]))
print(f"\nTraining final model on 100% of data with n_estimators={best_n_estimators}")

final_params = xgb_params.copy()
final_params.pop("early_stopping_rounds")
final_params.pop("eval_metric")
final_params["n_estimators"] = best_n_estimators

final_model = xgb.XGBRegressor(**final_params)
final_model.fit(X, y, verbose=False)


Training final model on 100% of data with n_estimators=300


,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [17]:
# Save the final model and metadata for later use
joblib.dump(final_model, "freight_model.joblib")

metadata = {
    "feature_cols": feature_cols,
    "weight_median": float(weight_median),
    "market_index_median": float(market_index_median),
}

metadata["equip_categories"] = sorted(df["equipment"].unique().tolist())

with open("model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)


print("\nSaved: freight_model.joblib, model_metadata.json")


Saved: freight_model.joblib, model_metadata.json


## Predict

In [18]:
import pandas as pd
import numpy as np
import joblib
import json

MODEL_PATH = "freight_model.joblib"
METADATA_PATH = "model_metadata.json"
INPUT_PATH = "validation.csv"
OUTPUT_PATH = "validation_predictions.csv"
df = pd.read_csv(INPUT_PATH, parse_dates=["date"])
print(f"Loaded {len(df)} validation loads")

Loaded 12000 validation loads


In [19]:
df.head()

,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal
0,TE-000001,Baton Rouge,Mobile,30.50134,-92.65374,31.80442,-88.25195,331.4,Flatbed,19958.0,2025-11-01,0.90402,1.86388
1,TE-000002,Savannah,Los Angeles,33.60973,-82.29994,28.56624,-116.70249,2406.2,Reefer,34114.0,2025-11-01,0.92851,1.86240
2,TE-000003,San Francisco,Raleigh,35.19670,-121.69849,35.37376,-77.47605,2846.6,Dry Van,33435.0,2025-11-01,0.88117,1.95480
3,TE-000004,Dayton,Los Angeles,39.87206,-85.62931,28.56624,-116.70249,2261.6,Dry Van,25392.0,2025-11-01,0.89156,2.14579
4,TE-000005,Memphis,San Antonio,35.56802,-89.52871,29.50969,-98.40059,800.4,Flatbed,NaN,2025-11-01,0.88308,2.22541


In [20]:
df.isnull().sum()

load_id           0
pickup            0
delivery          0
pickup_lat        0
pickup_lon        0
delivery_lat      0
delivery_lon      0
distance          0
equipment         0
weight          165
date              0
market_index    249
quote_signal      0
dtype: int64

In [21]:
model = joblib.load(MODEL_PATH)
with open(METADATA_PATH) as f:
    meta = json.load(f)

feature_cols = meta["feature_cols"]
weight_median = meta["weight_median"]
market_index_median = meta["market_index_median"]
equip_categories = meta["equip_categories"]
load_ids = df["load_id"].copy()

In [22]:
def engineer_features(df):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df["day_of_week"] = df["date"].dt.dayofweek
    df["day_of_month"] = df["date"].dt.day
    df["month"] = df["date"].dt.month
    df["log_distance"] = np.log1p(df["distance"])

    for cat in equip_categories:
        df[f"equip_{cat}"] = (df["equipment"] == cat).astype(int)

    return df

df_feat = engineer_features(df)

In [23]:

df_feat["weight_was_missing"] = df["weight"].isna().astype(int)
df_feat["market_index_was_missing"] = df["market_index"].isna().astype(int)
df_feat["weight"] = df_feat["weight"].fillna(weight_median)
df_feat["market_index"] = df_feat["market_index"].fillna(market_index_median)

In [24]:
X_val = df_feat[feature_cols]
pred_log = model.predict(X_val)
pred_dollars = np.expm1(pred_log)

In [25]:
out = pd.DataFrame({"load_id": load_ids, "predicted_rate": np.round(pred_dollars, 2)})
out.to_csv(OUTPUT_PATH, index=False)

print(f"Saved {len(out)} predictions to {OUTPUT_PATH}")
print(out.head())
print(f"\nPrediction summary: mean=${out['predicted_rate'].mean():,.2f}  "
      f"median=${out['predicted_rate'].median():,.2f}  "
      f"min=${out['predicted_rate'].min():,.2f}  max=${out['predicted_rate'].max():,.2f}")

Saved 12000 predictions to validation_predictions.csv
     load_id  predicted_rate
0  TE-000001      863.320007
1  TE-000002     4843.700195
2  TE-000003     5180.640137
3  TE-000004     4049.429932
4  TE-000005     1825.199951

Prediction summary: mean=$2,346.28  median=$2,032.01  min=$225.17  max=$7,169.57


## Running score.py

In [34]:
!python score.py --predictions validation_predictions.csv --december-predictions december-chart-inputs.csv

December predictions read successfully
ERROR: December predictions contains invalid predicted_rate values
